In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

In [0]:
dbutils.widgets.text("catalog_name", "")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
volume_path = f"/Volumes/{catalog_name}/lechster10_bronze/lab3_volume"
schema_directory = f"{volume_path}/schemas"
source_directory = f"{volume_path}/streaming_data"
checkpoint_location = f"{volume_path}/checkpoints"
target_table = "dbr_dev.lechster10_bronze.bitcoins_bronze"

### Autoloader & Structured Streaming

`Schema inference` <br>
.format("cloudFiles") \ <br>
.option("cloudFiles.schemaLocation", schema_directory) <br>
These lines enable smart schema reading. We also need to give location for storaging schema

"" <br>
To infer the schema when first reading data, Auto Loader samples the first 50 GB or 1000 files that it discovers, <br> whichever limit is crossed first. Auto Loader stores the schema information in a directory _schemas at the configured <br>`cloudFiles.schemaLocation` to track schema changes to the input data over time. <br > ""


"" <br>

The Apache Spark DataFrameReader uses different behavior for schema inference, selecting data types for columns in JSON, CSV, and XML <br> sources based on sample data. To enable this behavior with Auto Loader, set the option `cloudFiles.inferColumnTypes` to true <br > ""


### Reading and writing files with defining schema structure

In [0]:
# schema = StructType([
#     StructField("unix", IntegerType(), True),
#     StructField("date", TimestampType(), True),
#     StructField("symbol", StringType(), True),
#     StructField("open", DoubleType(), True),
#     StructField("high", DoubleType(), True),
#     StructField("low", DoubleType(), True),
#     StructField("close", DoubleType(), True),
#     StructField("Volume_BTC", DoubleType(), True),
#     StructField("Volume_USD", DoubleType(), True),
# ])

# stream_df = spark.readStream \
#     .format("cloudFiles") \
#     .schema(schema) \
#     .option("header", "true") \
#     .option("cloudFiles.format", "csv") \
#     .option("cloudFiles.schemaEvolutionMode", "rescue") \
#     .option("cloudFiles.schemaLocation", schema_directory) \
#     .load(source_directory)  

# stream_query = stream_df.writeStream \
#   .format("delta") \
#   .option("checkpointLocation", checkpoint_location) \
#   .trigger(availableNow=True) \
#   .table(target_table)

### Reading files without defining schema

In [0]:
# .option("cloudFiles.schemaEvolutionMode", "addNewColumnsWithTypeWidening") supports type widening

stream_df = spark.readStream.format("cloudFiles") \
    .option("header", "true")
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.maxFilesPerTrigger", 100) \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumnsWithTypeWidening") \
    .option("cloudFiles.schemaLocation", schema_directory) \
    .load(source_directory)

### Analyzing stats - trigger once 
`.trigger(Once=True)` - does not divide files in our folder into micro batches so 1000 files will be loaded straight away <br>
(if I use this option i need to delete .option("maxFilesPerTrigger", 100) from readStream)

![image_1784831134564.png](./image_1784831134564.png "image_1784831134564.png")

In [0]:
# stream_query = stream_df.writeStream \
#   .format("delta") \
#   .option("checkpointLocation", checkpoint_location) \
#   .trigger(once=True) \
#   .table(target_table)

### Analyzing stats - trigger availableNow
`.trigger(availableNow=True)` - divide files in our folder into micro batches (I can set max Files per Trigger to limit ingestion)
![image_1784830472982.png](./image_1784830472982.png "image_1784830472982.png")

In [0]:
stream_query = stream_df.writeStream \
  .format("delta") \
  .option("checkpointLocation", checkpoint_location) \
  .trigger(availableNow=True) \
  .option("mergeSchema", "true") \
  .table(target_table)

In [0]:
# display(spark.table(target_table))

#### Adding test column named "test" with value 1

In [0]:
# my_cols = ",".join(stream_df.columns[:-1]) + ",test"
# line = "1704067200,2024-01-01T00:00:00.000+00:00,BTC/USD,42350.75,43120.5,41890.2,42980.6,15234.892371,652481039.47,1"
# whole_line = my_cols + "\n" + line
# nr_of_row = 1001

# dbutils.fs.put(f"{volume_path}/streaming_data/file_{nr_of_row}.csv", whole_line, overwrite=True)

#### Trying to load file with extra column

new schema is added to schemas_

In [0]:
# stream_query = stream_df.writeStream \
#   .format("delta") \
#   .option("checkpointLocation", checkpoint_location) \
#   .trigger(availableNow=True) \
#   .option("mergeSchema", "true") \
#   .table(target_table)

In [0]:
# display(spark.table(target_table))

Finally, we can that all the rows in test column get null and the last one gets 1

### _rescued_data
Just showing it worked (it is not visible in original data because its impossible to change between addNewColumnsWithTypeWidening and rescue mode)

![image_1784834435423.png](./image_1784834435423.png "image_1784834435423.png")

In [0]:
# my_cols = ",".join(stream_df.columns[:-2]) + ",other"
# line = line = "1704153600,2024-01-02T00:00:00.000+00:00,BTC/USD,42980.6,44210.3,42650.15,43875.9,18921.457203,test"
# whole_line = my_cols + "\n" + line
# nr_of_row = 1004

# dbutils.fs.put(f"{volume_path}/streaming_data/file_{nr_of_row}.csv", whole_line, overwrite=True)

In [0]:
# stream_df = spark.readStream.format("cloudFiles") \
#     .option("cloudFiles.format", "csv") \
#     .option("cloudFiles.inferColumnTypes", "true") \
#     .option("cloudFiles.maxFilesPerTrigger", 100) \
#     .option("cloudFiles.schemaEvolutionMode", "rescue") \
#     .option("cloudFiles.schemaLocation", schema_directory) \
#     .load(source_directory)

In [0]:
# stream_query = stream_df.writeStream \
#   .format("delta") \
#   .option("checkpointLocation", checkpoint_location) \
#   .trigger(availableNow=True) \
#   .table(target_table)